# Lab 2 — Bias–Variance, Learning Curves và Model Selection

**Thời lượng:** 150–180 phút  
**Protocol:** split một lần → chọn degree bằng validation → learning-curve diagnosis → đóng băng quyết định → mở test một lần.

> Không thêm test metric vào degree-comparison table.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
OUTPUT = ROOT / 'outputs'
FIGURES = OUTPUT / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
np.set_printoptions(precision=6, suppress=True)
print('Project root:', ROOT.resolve())

## 1. Load data và khóa prediction contract

Model chỉ nhận một feature `x`; `sample_id` không phải feature.

In [ ]:
frame = pd.read_csv(ROOT / 'data' / 'nonlinear_regression.csv')
X = frame[['x']].to_numpy(dtype=float)
y = frame['y'].to_numpy(dtype=float)
assert X.shape == (300, 1)
assert y.shape == (300,)
assert frame['sample_id'].is_unique
frame.head()

## 2. Three-way split và invariants

Rounding rule: floor train, floor validation, phần còn lại là test. Test target không được truy cập ở các phần 3–6.

In [ ]:
def three_way_split_indices(n, train_fraction=0.6, validation_fraction=0.2, random_state=42):
    n_train = int(np.floor(train_fraction * n))
    n_validation = int(np.floor(validation_fraction * n))
    permutation = np.random.default_rng(random_state).permutation(n)
    train = permutation[:n_train]
    validation = permutation[n_train:n_train + n_validation]
    test = permutation[n_train + n_validation:]
    return train, validation, test


train_idx, validation_idx, test_idx = three_way_split_indices(len(frame))
all_indices = np.concatenate([train_idx, validation_idx, test_idx])
assert len(np.unique(all_indices)) == len(frame)
np.testing.assert_array_equal(np.sort(all_indices), np.arange(len(frame)))
assert (len(train_idx), len(validation_idx), len(test_idx)) == (180, 60, 60)
print('split sizes:', len(train_idx), len(validation_idx), len(test_idx))
test_access_count = 0

## 3. Polynomial pipeline và metric

Pipeline fit lại PolynomialFeatures/scaler/model chỉ trên training rows của từng experiment.

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_pred) - np.asarray(y_true)) ** 2)))


def polynomial_model(degree):
    return make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        StandardScaler(),
        LinearRegression(),
    )

## 4. Nhìn trực quan ba complexity levels

Chỉ vẽ train/validation observations. Test vẫn khóa.

In [ ]:
x_grid = np.linspace(X[:, 0].min(), X[:, 0].max(), 500).reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, degree in zip(axes, [1, 5, 15]):
    model = polynomial_model(degree).fit(X[train_idx], y[train_idx])
    ax.scatter(X[train_idx, 0], y[train_idx], s=14, alpha=0.45, label='Train')
    ax.scatter(X[validation_idx, 0], y[validation_idx], s=18, alpha=0.55, label='Validation')
    ax.plot(x_grid[:, 0], model.predict(x_grid), color='crimson')
    ax.set(title=f'Degree {degree}', xlabel='x')
axes[0].set_ylabel('y')
axes[0].legend()
fig.suptitle('Polynomial complexity — test set not shown')
fig.tight_layout()
fig.savefig(FIGURES / 'polynomial_fits.png', dpi=150)
plt.show()

## 5. Validation curve: degree 1–15

Train RMSE không phải selection metric. Chọn minimum validation RMSE; nếu hòa chọn degree nhỏ hơn.

In [ ]:
degree_rows = []
for degree in range(1, 16):
    model = polynomial_model(degree).fit(X[train_idx], y[train_idx])
    degree_rows.append({
        'degree': degree,
        'train_rmse': rmse(y[train_idx], model.predict(X[train_idx])),
        'validation_rmse': rmse(y[validation_idx], model.predict(X[validation_idx])),
    })
degree_comparison = pd.DataFrame(degree_rows)
degree_comparison.to_csv(OUTPUT / 'degree_comparison.csv', index=False)
degree_comparison

In [ ]:
selected_degree = int(
    degree_comparison.sort_values(['validation_rmse', 'degree']).iloc[0]['degree']
)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(degree_comparison['degree'], degree_comparison['train_rmse'], 'o-', label='Train')
ax.plot(degree_comparison['degree'], degree_comparison['validation_rmse'], 'o-', label='Validation')
ax.axvline(selected_degree, color='black', linestyle='--', label=f'Selected={selected_degree}')
ax.set(xlabel='Polynomial degree', ylabel='RMSE', title='Validation curve')
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES / 'degree_validation_curve.png', dpi=150)
plt.show()
print('Selected degree from validation:', selected_degree)
assert selected_degree == 10

## 6. Learning curves

Dùng một permutation của train để tạo nested subsets. Validation cố định; pipeline fit mới ở mọi điểm.

In [ ]:
def learning_curve_for_degree(degree, sizes, random_state=42):
    permutation = np.random.default_rng(random_state).permutation(len(train_idx))
    rows = []
    for size in sizes:
        subset_idx = train_idx[permutation[:size]]
        model = polynomial_model(degree).fit(X[subset_idx], y[subset_idx])
        rows.append({
            'degree': degree,
            'train_size': size,
            'train_rmse': rmse(y[subset_idx], model.predict(X[subset_idx])),
            'validation_rmse': rmse(y[validation_idx], model.predict(X[validation_idx])),
        })
    return pd.DataFrame(rows)


train_sizes = [30, 60, 90, 120, 180]
learning_curves = pd.concat(
    [learning_curve_for_degree(degree, train_sizes) for degree in [1, selected_degree, 15]],
    ignore_index=True,
)
selected_curve = learning_curves.query('degree == @selected_degree').drop(columns='degree')
selected_curve.to_csv(OUTPUT / 'learning_curve.csv', index=False)
learning_curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, degree in zip(axes, [1, selected_degree, 15]):
    curve = learning_curves.query('degree == @degree')
    ax.plot(curve['train_size'], curve['train_rmse'], 'o-', label='Train')
    ax.plot(curve['train_size'], curve['validation_rmse'], 'o-', label='Validation')
    ax.set(title=f'Degree {degree}', xlabel='Training rows')
    ax.grid(alpha=0.25)
axes[0].set_ylabel('RMSE')
axes[0].legend()
fig.suptitle('Learning curves — test remains locked')
fig.tight_layout()
fig.savefig(FIGURES / 'learning_curves.png', dpi=150)
plt.show()

## 7. Diagnosis trước test

- Degree 1: train và validation RMSE cùng cao → high bias/underfit; thêm data khó sửa model form.
- Degree 10: validation cải thiện khi thêm data, gap cuối nhỏ → candidate hợp lý.
- Degree 15: train tiếp tục tốt hơn nhưng validation không thắng degree 10 → complexity tăng không đem lại generalization gain.

**Quyết định đóng băng:** degree 10, pipeline không đổi, refit train+validation.

## 8. Leakage audit trước khi mở test

- PolynomialFeatures và scaler nằm trong Pipeline, fit riêng trên train/subset.
- `sample_id` không phải feature.
- Test không xuất hiện trong comparison/learning curves.
- Không chọn seed/degree bằng test.
- Split indices disjoint và exhaustive.

## 9. Final test — mở đúng một lần

Từ đây model decision đã khóa. Test được dùng để tính final estimate và error analysis, không để quay lại chọn degree.

In [ ]:
development_idx = np.concatenate([train_idx, validation_idx])
final_model = polynomial_model(selected_degree).fit(X[development_idx], y[development_idx])
test_access_count += 1
test_y = y[test_idx]
test_predictions = final_model.predict(X[test_idx])
test_rmse = rmse(test_y, test_predictions)
test_mae = float(np.mean(np.abs(test_predictions - test_y)))
test_r2 = float(1 - np.sum((test_predictions - test_y) ** 2) / np.sum((test_y - test_y.mean()) ** 2))
final_metrics = {
    'selected_degree': selected_degree,
    'test_rmse': test_rmse,
    'test_mae': test_mae,
    'test_r2': test_r2,
    'test_rows': len(test_idx),
    'test_evaluation_count': test_access_count,
}
(OUTPUT / 'final_test_metrics.json').write_text(json.dumps(final_metrics, indent=2), encoding='utf-8')
print(json.dumps(final_metrics, indent=2))

In [ ]:
prediction_frame = pd.DataFrame({
    'sample_id': frame.iloc[test_idx]['sample_id'].to_numpy(),
    'actual_y': test_y,
    'predicted_y': test_predictions,
    'residual': test_predictions - test_y,
})
prediction_frame.to_csv(OUTPUT / 'nonlinear_test_predictions.csv', index=False)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(test_y, test_predictions, alpha=0.7)
limits = [min(test_y.min(), test_predictions.min()), max(test_y.max(), test_predictions.max())]
ax.plot(limits, limits, '--', color='black', label='Perfect')
ax.set(xlabel='Actual y', ylabel='Predicted y', title='Final test: predicted vs actual')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / 'predicted_vs_actual.png', dpi=150)
plt.show()

## Exit ticket

1. Tại sao degree 15 không được chọn dù train RMSE thấp hơn?
2. Learning curve degree 1 cho biết thêm data có giúp nhiều không?
3. Nếu đổi degree sau khi xem test RMSE, test set đổi vai trò thành gì?
4. Nếu rows thuộc cùng customer, random row split cần thay bằng gì?

In [ ]:
assert test_access_count == 1
assert selected_degree == 10
assert abs(test_rmse - 1.2247055846717751) < 1e-10
assert set(degree_comparison.columns) == {'degree', 'train_rmse', 'validation_rmse'}
assert len(prediction_frame) == 60
assert (FIGURES / 'learning_curves.png').exists()
print('PASS — split, validation-only selection, learning curves và final test đạt.')